# Readmission Risk Modeling: EDA and Baseline Pipeline

**Digital Health & Medical Data Science Track**

This notebook covers Q1–Q4 with visualizations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## Load Dataset

In [ ]:
# Load dataset
df = pd.read_csv('diabetic_data.csv')
print(f"Dataset Shape: {df.shape}")

## Q1: Readmission Population Overview

In [ ]:
print("\n--- Q1: Target Distribution ---")
print(df['readmitted'].value_counts())

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='readmitted', order=['NO', '>30', '<30'], palette='Set2')
plt.title('Q1: Distribution of Readmission Classes', fontsize=14, fontweight='bold')
plt.xlabel('Readmission Status', fontsize=12)
plt.ylabel('Count (Encounters)', fontsize=12)
plt.show()

In [ ]:
# Replace '?' with NaN and check weight missingness
df = df.replace('?', np.nan)

## Q2: Measurements across Readmission Groups

In [ ]:
metrics = [
    'time_in_hospital',
    'num_medications',
    'number_inpatient'
]

print("\n--- Q2: Summary Statistics Across Readmission Groups ---")
for metric in metrics:
    print(f"\nMetric: {metric}")
    print(df.groupby('readmitted')[metric].describe())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, metric in enumerate(metrics):
    sns.boxplot(data=df, x='readmitted', y=metric, order=['NO', '>30', '<30'], ax=axes[i], palette='Pastel1')
    axes[i].set_title(f'Q2: {metric} by Readmission', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Q3: Variables Associated with 30-Day Readmission

In [ ]:
df['target_early_readmit'] = (df['readmitted'] == '<30').astype(int)

print("\n--- Q3: Binary Early Readmission Distribution ---")
print(df['target_early_readmit'].value_counts(normalize=True) * 100)

plt.figure(figsize=(7, 5))
sns.barplot(data=df, x='target_early_readmit', y='number_inpatient', palette='viridis', ci=None)
plt.title('Q3: Prior Inpatient Visits vs Early Readmission', fontsize=14, fontweight='bold')
plt.xlabel('Early Readmission (<30 days: 1, Rest: 0)', fontsize=12)
plt.ylabel('Average Prior Inpatient Visits', fontsize=12)
plt.show()

## Q4: Minimal Preprocessing & Baseline Pipeline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

numeric_features = ['time_in_hospital', 'num_lab_procedures', 'num_medications', 'number_inpatient', 'number_emergency']
categorical_features = ['race', 'gender', 'age']

model_cols = numeric_features + categorical_features + ['patient_nbr', 'target_early_readmit']
df_model = df[model_cols].dropna(subset=['target_early_readmit'])

# Patient-level split
patients = df_model['patient_nbr'].unique()
train_patients, test_patients = train_test_split(patients, test_size=0.2, random_state=42)

train_df = df_model[df_model['patient_nbr'].isin(train_patients)]
test_df = df_model[df_model['patient_nbr'].isin(test_patients)]

X_train = train_df[numeric_features + categorical_features]
y_train = train_df['target_early_readmit']
X_test = test_df[numeric_features + categorical_features]
y_test = test_df['target_early_readmit']

# Preprocessing & Pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

baseline_pipeline.fit(X_train, y_train)
y_pred = baseline_pipeline.predict(X_test)

print("\n--- Q4: Baseline Pipeline Classification Report ---")
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Q4: Baseline Model Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.show()